In [0]:
# ==========================================
# 02. DATA CLEANING & DEDUPLICATION
# ==========================================
from pyspark.sql.functions import col, trim, current_timestamp, row_number
from pyspark.sql.window import Window

catalog = "workspace"
schema_name = "default"
bronze_table = f"{catalog}.{schema_name}.wiki_bronze_data"

# 1. Read raw Bronze data
df_bronze = spark.table(bronze_table)

# 2. Filter invalid records and clean strings
df_cleaned = df_bronze \
    .filter(col("id").isNotNull()) \
    .withColumn("title", trim(col("title"))) \
    .withColumn("user", trim(col("user"))) \
    .withColumn("comment", trim(col("comment"))) \
    .withColumn("_ingestion_time", current_timestamp())

# 3. Deduplicate using Window specification
window_spec = Window.partitionBy("id", "timestamp").orderBy(col("_ingestion_time").desc())

df_deduplicated = df_cleaned \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

print("✅ Step 2 Complete! Data cleaned and deduplicated.")
display(df_deduplicated)